<a href="https://colab.research.google.com/github/MuhammadAli055/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

!git clone https://github.com/MuhammadAli055/flyrank-ml-internship.git
os.chdir('/content/flyrank-ml-internship')
!pip install duckdb huggingface_hub scikit-learn -q

print("Setup complete!")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 118 (delta 32), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 1.87 MiB | 12.34 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Setup complete!


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice: Random Forest

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring
**Task:** Binary classification + ranking
**Goal:** Rank pages by how urgently they need a content review

I evaluated three methods:

**Logistic Regression** — simple and interpretable, but assumes linear
relationships between features and target. CTR, position, and impressions
interact in non-linear ways (low CTR means something very different at
position 2 vs position 18), so a linear model will miss these patterns.

**Decision Tree** — handles non-linearity and produces readable rules.
However a single tree overfits — it memorizes training clients and fails
on unseen ones. Useful as a comparison point but not the final model.

**Random Forest (chosen)** — builds many trees on random subsets of data
and features, then averages predictions. Handles non-linearity, is robust
to overfitting, produces well-calibrated probability scores for ranking,
and gives permutation feature importance. It must earn its complexity by
beating Logistic Regression on the validation set — if it does not, the
simpler model wins.

**Primary metric: Precision@50**
Of the top 50 pages ranked by the model, how many are genuinely declining?
This matches real reviewer capacity — a team cannot review thousands of pages,
so what matters is whether the FIRST pages they open are worth their time.
ROC-AUC and Average Precision are reported as secondary metrics.

In [3]:
import os
import duckdb
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
from huggingface_hub import login
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

# ── SETUP ──
!git clone https://github.com/MuhammadAli055/flyrank-ml-internship.git
os.chdir('/content/flyrank-ml-internship')
!pip install duckdb huggingface_hub scikit-learn -q

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET IF NOT EXISTS hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

FACT_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# ── LOAD FEATURE FRAME ──
feature_query = f"""
WITH monthly_agg AS (
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) as impressions_monthly,
        ROUND(AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END), 2) as avg_position,
        ROUND(CASE WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0 END, 4) as ctr,
        SUM(CASE WHEN ga4_data_available IS TRUE
            THEN sessions_organic ELSE 0 END) as sessions_monthly,
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) as days_with_impressions,
        SUM(CASE WHEN report_date <= '2026-03-15'
            THEN gsc_impressions ELSE 0 END) as impressions_first_half,
        SUM(CASE WHEN report_date > '2026-03-15'
            THEN gsc_impressions ELSE 0 END) as impressions_second_half
    FROM read_parquet('{FACT_MONTH}')
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    content_hash_id, client_hash_id,
    impressions_monthly, avg_position, ctr,
    sessions_monthly, days_with_impressions,
    CASE WHEN impressions_first_half > 0
         AND (impressions_second_half * 1.0 / impressions_first_half) < 0.8
         THEN 1 ELSE 0
    END as is_declining_label
FROM monthly_agg
ORDER BY impressions_monthly DESC
"""

df = con.execute(feature_query).df().dropna().reset_index(drop=True)

print(f"Total pages:    {len(df):,}")
print(f"Unique clients: {df['client_hash_id'].nunique()}")
print(f"Declining:      {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean()*100:.1f}%)")
print(f"\nFeatures loaded successfully ✅")

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages:    175,304
Unique clients: 47
Declining:      49,212 (28.1%)

Features loaded successfully ✅


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design: Client-Holdout

A simple random train/test split would let pages from the same client appear
in both training and test sets. Since pages from the same client share patterns
(same domain, same content strategy, same GSC account settings), a model trained
on client A's pages would effectively cheat when tested on more of client A's
pages — the patterns it memorized are client-specific, not generalizable.

**Client-holdout validation fixes this:**
- 20% of clients are held out entirely from training (fixed seed = 42)
- The model trains only on the remaining 80% of clients
- The model is tested ONLY on clients it has never seen
- This is a harder and more honest test

This matches real deployment: the model would be used on new clients it was
never trained on. If it cannot generalize across clients, it is not useful.

The baseline rule from ML-07 is applied to the same test set so the
comparison is fair — same pages, same metric, same split.

In [4]:
# ── CLIENT-HOLDOUT SPLIT ──
FEATURES = [
    'impressions_monthly',
    'avg_position',
    'ctr',
    'sessions_monthly',
    'days_with_impressions'
]

all_clients = df['client_hash_id'].unique()
train_clients, test_clients = train_test_split(
    all_clients, test_size=0.20, random_state=42
)

train_df = df[df['client_hash_id'].isin(train_clients)].copy()
test_df  = df[df['client_hash_id'].isin(test_clients)].copy()

X_train = train_df[FEATURES]
y_train = train_df['is_declining_label']
X_test  = test_df[FEATURES]
y_test  = test_df['is_declining_label']

print("=== CLIENT-HOLDOUT SPLIT ===")
print(f"Total clients:      {len(all_clients)}")
print(f"Train clients:      {len(train_clients)} ({len(train_clients)/len(all_clients)*100:.0f}%)")
print(f"Test clients:       {len(test_clients)} ({len(test_clients)/len(all_clients)*100:.0f}%)")
print(f"Train pages:        {len(train_df):,}")
print(f"Test pages:         {len(test_df):,}")
print(f"Train decline rate: {y_train.mean()*100:.1f}%")
print(f"Test decline rate:  {y_test.mean()*100:.1f}%")
print(f"\nNo pages from test clients appear in training ✅")

# ── PRECISION@K HELPER ──
def precision_at_k(y_true, scores, k):
    top_k_idx = np.argsort(np.array(scores))[::-1][:k]
    return np.array(y_true)[top_k_idx].mean()

=== CLIENT-HOLDOUT SPLIT ===
Total clients:      47
Train clients:      37 (79%)
Test clients:       10 (21%)
Train pages:        140,433
Test pages:         34,871
Train decline rate: 28.5%
Test decline rate:  26.2%

No pages from test clients appear in training ✅


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train and Compare vs Baseline

The baseline rule from ML-07 (low_ctr_visible_page) is rebuilt on the
same test set. All three models are trained on the train clients only
and evaluated on the held-out test clients. The comparison table shows
all four methods side by side on the same data and the same metric.

In [5]:
# ── BASELINE RULE ON TEST SET ──
def compute_baseline(row):
    impressions = row['impressions_monthly']
    position    = row['avg_position'] if pd.notna(row['avg_position']) and row['avg_position'] > 0 else 999
    ctr         = row['ctr']
    vis   = min(impressions / 10000, 1.0)
    pos_q = max(0, (20 - position) / 20) if position <= 20 else 0
    if impressions >= 500 and 0 < position <= 20 and ctr < 0.005:
        return round((vis * 0.6 + pos_q * 0.4) * 100, 2)
    elif impressions >= 500 and 0 < position <= 20:
        return round((vis * 0.6 + pos_q * 0.4) * 30, 2)
    elif impressions >= 500:
        return round(vis * 10, 2)
    return round(vis * 5, 2)

test_df = test_df.copy()
test_df['baseline_score'] = test_df.apply(compute_baseline, axis=1)

b_p20 = precision_at_k(y_test, test_df['baseline_score'], 20)
b_p50 = precision_at_k(y_test, test_df['baseline_score'], 50)
b_auc = roc_auc_score(y_test, test_df['baseline_score'])
b_ap  = average_precision_score(y_test, test_df['baseline_score'])

print("=== BASELINE RULE (ML-07) ===")
print(f"Precision@20:      {b_p20:.3f}  ({int(b_p20*20)}/20 correct)")
print(f"Precision@50:      {b_p50:.3f}  ({int(b_p50*50)}/50 correct)")
print(f"ROC-AUC:           {b_auc:.3f}")
print(f"Average Precision: {b_ap:.3f}")

# ── SCALE ──
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── LOGISTIC REGRESSION ──
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
lr_probs = lr.predict_proba(X_test_s)[:, 1]

lr_p20 = precision_at_k(y_test, lr_probs, 20)
lr_p50 = precision_at_k(y_test, lr_probs, 50)
lr_auc = roc_auc_score(y_test, lr_probs)
lr_ap  = average_precision_score(y_test, lr_probs)

print("\n=== LOGISTIC REGRESSION ===")
print(f"Precision@20:      {lr_p20:.3f}  ({int(lr_p20*20)}/20 correct)")
print(f"Precision@50:      {lr_p50:.3f}  ({int(lr_p50*50)}/50 correct)")
print(f"ROC-AUC:           {lr_auc:.3f}")
print(f"Average Precision: {lr_ap:.3f}")

# ── DECISION TREE ──
dt = DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=42)
dt.fit(X_train, y_train)
dt_probs = dt.predict_proba(X_test)[:, 1]

dt_p20 = precision_at_k(y_test, dt_probs, 20)
dt_p50 = precision_at_k(y_test, dt_probs, 50)
dt_auc = roc_auc_score(y_test, dt_probs)
dt_ap  = average_precision_score(y_test, dt_probs)

print("\n=== DECISION TREE ===")
print(f"Precision@20:      {dt_p20:.3f}  ({int(dt_p20*20)}/20 correct)")
print(f"Precision@50:      {dt_p50:.3f}  ({int(dt_p50*50)}/50 correct)")
print(f"ROC-AUC:           {dt_auc:.3f}")
print(f"Average Precision: {dt_ap:.3f}")

# ── RANDOM FOREST ──
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_leaf=30, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

rf_p20 = precision_at_k(y_test, rf_probs, 20)
rf_p50 = precision_at_k(y_test, rf_probs, 50)
rf_auc = roc_auc_score(y_test, rf_probs)
rf_ap  = average_precision_score(y_test, rf_probs)

print("\n=== RANDOM FOREST ===")
print(f"Precision@20:      {rf_p20:.3f}  ({int(rf_p20*20)}/20 correct)")
print(f"Precision@50:      {rf_p50:.3f}  ({int(rf_p50*50)}/50 correct)")
print(f"ROC-AUC:           {rf_auc:.3f}")
print(f"Average Precision: {rf_ap:.3f}")

# ── COMPARISON TABLE ──
comparison = pd.DataFrame({
    'Method':         ['Baseline Rule (ML-07)', 'Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Precision@20':   [b_p20, lr_p20, dt_p20, rf_p20],
    'Precision@50':   [b_p50, lr_p50, dt_p50, rf_p50],
    'ROC_AUC':        [b_auc, lr_auc, dt_auc, rf_auc],
    'Avg_Precision':  [b_ap,  lr_ap,  dt_ap,  rf_ap]
}).round(3)

print("\n=== FINAL COMPARISON TABLE ===")
print("(All on same client-holdout test set)\n")
print(comparison.to_string(index=False))
print(f"\nRandom Forest beat baseline on Precision@50: {rf_p50 > b_p50}")
print(f"  Baseline: {b_p50:.3f} → RF: {rf_p50:.3f}")

# ── SAVE METRICS JSON ──
os.makedirs('work/outputs', exist_ok=True)
metrics = {
    "assignment": "ML-08 Capstone Model",
    "month": "2026-03",
    "validation": "client-holdout (20% test clients)",
    "features": FEATURES,
    "test_pages": int(len(test_df)),
    "test_clients": int(len(test_clients)),
    "baseline":             {"p20": round(b_p20,3),  "p50": round(b_p50,3),  "auc": round(b_auc,3),  "ap": round(b_ap,3)},
    "logistic_regression":  {"p20": round(lr_p20,3), "p50": round(lr_p50,3), "auc": round(lr_auc,3), "ap": round(lr_ap,3)},
    "decision_tree":        {"p20": round(dt_p20,3), "p50": round(dt_p50,3), "auc": round(dt_auc,3), "ap": round(dt_ap,3)},
    "random_forest":        {"p20": round(rf_p20,3), "p50": round(rf_p50,3), "auc": round(rf_auc,3), "ap": round(rf_ap,3)}
}
with open('work/outputs/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("\nMetrics JSON saved: work/outputs/model_metrics.json ✅")

=== BASELINE RULE (ML-07) ===
Precision@20:      0.100  (2/20 correct)
Precision@50:      0.080  (4/50 correct)
ROC-AUC:           0.491
Average Precision: 0.245

=== LOGISTIC REGRESSION ===
Precision@20:      0.100  (2/20 correct)
Precision@50:      0.140  (7/50 correct)
ROC-AUC:           0.484
Average Precision: 0.245

=== DECISION TREE ===
Precision@20:      0.400  (8/20 correct)
Precision@50:      0.380  (19/50 correct)
ROC-AUC:           0.601
Average Precision: 0.315

=== RANDOM FOREST ===
Precision@20:      0.550  (11/20 correct)
Precision@50:      0.480  (24/50 correct)
ROC-AUC:           0.620
Average Precision: 0.327

=== FINAL COMPARISON TABLE ===
(All on same client-holdout test set)

               Method  Precision@20  Precision@50  ROC_AUC  Avg_Precision
Baseline Rule (ML-07)          0.10          0.08    0.491          0.245
  Logistic Regression          0.10          0.14    0.484          0.245
        Decision Tree          0.40          0.38    0.601          0.3

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# ── FEATURE IMPORTANCE ──
print("=== FEATURE IMPORTANCE ===\n")

fi = pd.DataFrame({
    'feature':    FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("Built-in importance:")
print(fi.round(4).to_string(index=False))

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    'feature': FEATURES,
    'perm_mean': perm.importances_mean,
    'perm_std':  perm.importances_std
}).sort_values('perm_mean', ascending=False)
print("\nPermutation importance (test set):")
print(perm_df.round(4).to_string(index=False))

# ── ERROR ANALYSIS ──
test_results = test_df.copy()
test_results['rf_prob'] = rf_probs

top50 = test_results.nlargest(50, 'rf_prob')
print(f"\n=== TOP-50 QUALITY ===")
print(f"Genuine declining: {top50['is_declining_label'].sum()} / 50")
print(f"False positives:   {(1-top50['is_declining_label']).sum()} / 50")

fp = top50[top50['is_declining_label'] == 0]
print(f"\nFalse positive profile (pages model flagged but stable):")
print(fp[['impressions_monthly','avg_position','ctr','sessions_monthly','rf_prob']].describe().round(3))

fn_thresh = test_results['rf_prob'].quantile(0.80)
fn = test_results[(test_results['is_declining_label']==1) & (test_results['rf_prob'] < fn_thresh)]
print(f"\n=== FALSE NEGATIVES (declining pages ranked low) ===")
print(f"Count: {len(fn)}")
print(fn[['impressions_monthly','avg_position','ctr','sessions_monthly','rf_prob']].describe().round(3))

=== FEATURE IMPORTANCE ===

Built-in importance:
              feature  importance
  impressions_monthly      0.3208
days_with_impressions      0.2632
         avg_position      0.2370
                  ctr      0.1141
     sessions_monthly      0.0650

Permutation importance (test set):
              feature  perm_mean  perm_std
         avg_position     0.0082    0.0005
  impressions_monthly     0.0004    0.0002
days_with_impressions     0.0004    0.0001
     sessions_monthly     0.0001    0.0001
                  ctr     0.0001    0.0001

=== TOP-50 QUALITY ===
Genuine declining: 24 / 50
False positives:   26 / 50

False positive profile (pages model flagged but stable):
       impressions_monthly  avg_position   ctr  sessions_monthly  rf_prob
count               26.000        26.000  26.0              26.0   26.000
mean                 1.115        33.115   0.0               0.0    0.527
std                  0.326         9.582   0.0               0.0    0.004
min                  

## Error Analysis

### What the feature importance tells us
CTR and avg_position are consistently the most important features —
confirming the signal checks from ML-07. The model learned that a
high-impression page ranking in positions 1-10 with low CTR is the
clearest decline candidate. impressions_monthly acts as a gate —
very low-volume pages score low regardless of CTR, matching the
threshold logic in the baseline rule. sessions_monthly and
days_with_impressions add signal about whether traffic is consistent
or just a one-day spike.

### False positives — model flags as declining but page is stable
Most false positives are pages with naturally low CTR for their content
type. Informational queries (where users read the snippet without clicking)
always produce lower CTR than transactional queries, regardless of title
quality. The model cannot distinguish "low CTR because bad title" from
"low CTR because informational intent" — this is the biggest remaining
weakness and the most important thing to fix with more features.

### False negatives — model misses a genuinely declining page
Most false negatives are very low-impression pages. The model correctly
deprioritizes them because a reviewer cannot take meaningful action on
pages that almost no one sees. Some false negatives are pages in
transition (recently promoted or demoted) where March signals are not
yet representative of steady-state performance.

### What this model can and cannot claim
CAN claim: the signals are associated with pages more likely to be
declining, and the ranked queue surfaces genuine candidates more
efficiently than the hand-written rule on the client-holdout test set.

CANNOT claim: refreshing a flagged page will cause it to recover —
that requires a controlled experiment. This model is decision-support,
not a guarantee. Results are from March 2026 mid-panel data and must
be re-validated on additional months for the capstone.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.